In [ ]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080220

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")


## 一、围绕半圆定律（Wigner 随机矩阵）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False


def wigner_density(x):
    """半圆密度 f(x)=sqrt(4-x^2)/(2pi)，|x|<=2；否则为 0。兼容标量和数组。"""
    x_arr = np.asarray(x, dtype=float)
    inside = np.abs(x_arr) <= 2
    radicand = np.maximum(4 - x_arr ** 2, 0)
    val = np.where(inside, np.sqrt(radicand) / (2 * np.pi), 0.0)
    return float(val) if val.ndim == 0 else val


_rng = np.random.default_rng(0)


def generate_wigner(n):
    """生成 n×n 对称 Wigner 矩阵：上三角独立抽样，再对称填充下三角。"""
    if isinstance(n, bool) or not isinstance(n, int):
        raise TypeError("n 必须是整数")
    if n <= 0:
        raise ValueError("n 必须是正整数")
    upper = _rng.normal(0, np.sqrt(1 / n), size=(n, n))
    tri = np.triu(upper)
    return tri + tri.T - np.diag(np.diag(tri))


def get_spectrum(matrix):
    """返回全部特征值，按非递减顺序排列。"""
    eig = np.linalg.eigvalsh(matrix)
    return np.sort(eig)


print(wigner_density(0))
print(wigner_density(2))
print(wigner_density(3))
print(generate_wigner(3))
print(get_spectrum(generate_wigner(3)))


In [ ]:
# 四个纵向子图：n = 100, 200, 500, 1000
fig, axes = plt.subplots(4, 1, figsize=(8, 12))
xs = np.linspace(-2, 2, 200)
for ax, n in zip(axes, [100, 200, 500, 1000]):
    spectrum = get_spectrum(generate_wigner(n))
    ax.hist(spectrum, bins=40, density=True, color="blue", alpha=0.6, label="特征值直方图")
    ax.plot(xs, wigner_density(xs), color="red", label="半圆密度曲线")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("特征值")
    ax.set_ylabel("密度")
    ax.legend()
plt.tight_layout()
plt.show()


从图上可以看出：n=100、200 时直方图波动还比较大；**n 大约达到 500 时已能较好地拟合半圆密度**，
到 n=1000 时直方图形状与红色半圆曲线基本吻合，说明特征值分布随阶数增大趋近半圆定律。

## 二、Conway 生命游戏

In [ ]:
def is_valid_board(board):
    """有效棋盘：NumPy 二维数组，元素仅为 0 或 1。返回 Python 布尔值。"""
    if not isinstance(board, np.ndarray) or board.ndim != 2:
        return False
    return bool(np.all((board == 0) | (board == 1)))


def gol_step(board):
    """按环绕边界规则推进一步；邻居状态全部来自原棋盘。"""
    if not is_valid_board(board):
        raise ValueError("无效棋盘：必须是元素为 0/1 的二维 NumPy 数组")
    b = board.astype(int)
    neighbors = np.zeros_like(b)
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            neighbors += np.roll(np.roll(b, di, axis=0), dj, axis=1)
    nxt = np.zeros_like(b)
    survive = (b == 1) & ((neighbors == 2) | (neighbors == 3))
    born = (b == 0) & (neighbors == 3)
    nxt[survive | born] = 1
    return nxt


def draw_gol_board(board):
    """活细胞黑色、死细胞白色，不显示网格线、轴标签和刻度。"""
    cmap = plt.get_cmap("gray_r")
    plt.figure()
    plt.imshow(board, cmap=cmap, interpolation="nearest")
    plt.xticks([])
    plt.yticks([])
    plt.show()


In [ ]:
# 演示一个竖排闪烁器（blinker），放在 5×5 棋盘中央
board = np.array([[0, 0, 0, 0, 0],
                  [0, 0, 1, 0, 0],
                  [0, 0, 1, 0, 0],
                  [0, 0, 1, 0, 0],
                  [0, 0, 0, 0, 0]])
print("原棋盘:\n", board)
step1 = gol_step(board)
print("下一步:\n", step1)
print("是否有效棋盘:", is_valid_board(board))
draw_gol_board(board)
draw_gol_board(step1)


## 三、使用 pandas DataFrame（iris 数据集）

In [ ]:
import pandas as pd

iris = pd.read_csv("iris.csv")

# 统一列名，兼容 Kaggle 不同命名风格
rename = {}
for c in iris.columns:
    cl = c.lower()
    if "sepal" in cl and "length" in cl:
        rename[c] = "sepal_length"
    elif "sepal" in cl and "width" in cl:
        rename[c] = "sepal_width"
    elif "petal" in cl and "length" in cl:
        rename[c] = "petal_length"
    elif "petal" in cl and "width" in cl:
        rename[c] = "petal_width"
    elif "species" in cl or "class" in cl:
        rename[c] = "species"
iris = iris.rename(columns=rename)

print("数据点数:", len(iris))
print("列名:", list(iris.columns))
print("各列类型:\n", iris.dtypes)
print("花卉种类数:", iris["species"].nunique())
print("种类:", iris["species"].unique())


In [ ]:
# 修正第 35、38 行（0 起始索引 34、37）
iris.loc[34, ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]] = \
    [4.9, 3.1, 1.5, 0.2, "setosa"]
iris.loc[37, ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]] = \
    [4.9, 3.6, 1.4, 0.1, "setosa"]
print(iris.loc[[34, 37]])


In [ ]:
# 新增两个比例特征
iris["Petal.Ratio"] = iris["petal_length"] / iris["petal_width"]
iris["Sepal.Ratio"] = iris["sepal_length"] / iris["sepal_width"]

# 保存更正并扩展后的数据
iris.to_csv("iris_corrected.csv", index=False)
print(iris.head())


In [ ]:
# 按花卉种类分组，统计两个比例的均值、中位数、最小值、最大值、标准差
summary = iris.groupby("species")[["Petal.Ratio", "Sepal.Ratio"]].agg(
    ["mean", "median", "min", "max", "std"]
)
print(summary)
